# E09 Mall Customers：缩放、聚类数与稳定性

> Issue #9 / Epic #6 / 路线图 E09

## 你将学到
1. 聚类前必须缩放
2. k 的选择：肘部法 + silhouette
3. 多次随机种子检查簇稳定性

## 数据
- 本地 `data/Mall_Customers.csv`
- 回退：合成客户画像 DEMO


In [ ]:
QUICK = True
RANDOM_STATE = 42

import warnings
warnings.filterwarnings('ignore')

from wanglaoshi import VERSIONS as V
from wanglaoshi import JupyterEnv as JE

JE.no_warning()
V.show_env_info()


## 1. 加载或合成


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

path = Path('../../data/Mall_Customers.csv')
DEMO = False
if path.exists():
    df = pd.read_csv(path)
    print('local', df.shape)
else:
    DEMO = True
    rng = np.random.RandomState(RANDOM_STATE)
    n = 300
    df = pd.DataFrame({
        'CustomerID': np.arange(1, n + 1),
        'Genre': rng.choice(['Male', 'Female'], n),
        'Age': rng.randint(18, 70, n),
        'Annual Income (k$)': rng.normal(60, 25, n).clip(15, 140),
        'Spending Score (1-100)': rng.normal(50, 25, n).clip(1, 100),
    })
    print('DEMO customers', df.shape)
df.head()


## 2. EDA


In [ ]:
from wanglaoshi import DataAnalyzer

analyzer = DataAnalyzer(df)
display(analyzer.basic_statistics())


## 3. 选特征、缩放、搜 k


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# 常见列名兼容
cols = []
for a, b in [
    ('Annual Income (k$)', 'Annual Income (k$)'),
    ('Spending Score (1-100)', 'Spending Score (1-100)'),
    ('Age', 'Age'),
]:
    if a in df.columns:
        cols.append(a)
if len(cols) < 2:
    cols = df.select_dtypes(include=[np.number]).columns.tolist()[:3]
    cols = [c for c in cols if 'id' not in c.lower()]

X = df[cols].astype(float)
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

ks = range(2, 9)
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lab = km.fit_predict(Xs)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(Xs, lab))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].plot(list(ks), inertias, marker='o'); ax[0].set_title('Elbow'); ax[0].set_xlabel('k')
ax[1].plot(list(ks), sils, marker='o'); ax[1].set_title('Silhouette'); ax[1].set_xlabel('k')
plt.tight_layout(); plt.show()
best_k = list(ks)[int(np.argmax(sils))]
print('best_k by silhouette', best_k, 'score', max(sils))


## 4. 多随机种子稳定性


In [ ]:
from sklearn.metrics import adjusted_rand_score

base = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10).fit_predict(Xs)
aris = []
for seed in range(5 if QUICK else 20):
    lab = KMeans(n_clusters=best_k, random_state=seed, n_init=10).fit_predict(Xs)
    aris.append(adjusted_rand_score(base, lab))
print('ARI vs base seed: mean', np.mean(aris), 'min', np.min(aris), aris)


## 5. 画像解读


In [ ]:
labels = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10).fit_predict(Xs)
profile = df.copy()
profile['cluster'] = labels
display(profile.groupby('cluster')[cols].mean())


## 6. 检查清单
- [x] 缩放后聚类
- [x] elbow + silhouette
- [x] 多 seed ARI 稳定性
- [ ] 业务命名簇（高消费/低消费等）
- [ ] 与监督标签（若有）对照
